In [4]:
import pickle
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [3]:
with open("donnees_et_preprocesseur.pkl","rb") as f:
    X_train,X_test,y_train,y_test,preprocessor = pickle.load(f)

In [10]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modeles_a_optimiser = {
    "Logistic Regression": {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100] 
        }
    },
    "Random Forest": {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'classifier__n_estimators': [100, 200, 300],    
            'classifier__max_depth': [10, 20, 30, None],     
            'classifier__min_samples_split': [2, 5, 10]
        }
    },
    "XGBoost": {
        'model': XGBClassifier(random_state=42, eval_metric='mlogloss'),
        'params': {
            'classifier__n_estimators': [100, 200],          
            'classifier__learning_rate': [0.01, 0.05, 0.1],  
            'classifier__max_depth': [3, 5, 7]               
        }
    }
}

meilleurs_modeles = {}
resultats_comparaison = []

print("---  OPTIMISATION DES MODELES (GridSearchCV) ---")
for nom, config in modeles_a_optimiser.items():
    print(f"Optimisation en cours : {nom}...")
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', config['model'])
    ])
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=config['params'],
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train_encoded)
    
    meilleurs_modeles[nom] = grid_search.best_estimator_
    
    resultats_comparaison.append({
        "Modele": nom,
        "Meilleurs Parametres": grid_search.best_params_,
        "Score CV (Accuracy)": grid_search.best_score_
    })

df_resultats = pd.DataFrame(resultats_comparaison)
print("\n" + "="*50)
print(" COMPARISON DES MODELES (Validation Croisee)")
print("="*50)
print(df_resultats.to_string(index=False))


meilleur_nom_modele = df_resultats.loc[df_resultats["Score CV (Accuracy)"].idxmax()]["Modele"]
modele_optimal = meilleurs_modeles[meilleur_nom_modele]



---  OPTIMISATION DES MODELES (GridSearchCV) ---
Optimisation en cours : Logistic Regression...
Optimisation en cours : Random Forest...
Optimisation en cours : XGBoost...

 COMPARISON DES MODELES (Validation Croisee)
             Modele                                                                                Meilleurs Parametres  Score CV (Accuracy)
Logistic Regression                                                                                {'classifier__C': 1}              0.75250
      Random Forest {'classifier__max_depth': 20, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}              0.69125
            XGBoost    {'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 200}              0.69875
